In [1]:
# CELL 1 - CÀI THƯ VIỆN + MOUNT DRIVE + SPARK/DELTA
# [ĐỒNG]
# Mục đích:
# - Cài PySpark + Delta
# - Mount Drive
# - Khai báo đúng 4 thư mục cấp 1
# - Chuẩn bị SparkSession để ghi Delta tables và Spark ML models

!pip -q install pyspark delta-spark

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from delta import configure_spark_with_delta_pip

builder = (
    SparkSession.builder
    .appName("FIFA23_Lakehouse_Exact_Structure")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
)

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)

DATA_DIR = "/content/drive/MyDrive/FIFA23_BIGDATA"

BRONZE_DIR = os.path.join(DATA_DIR, "BRONZE")
SILVER_DIR = os.path.join(DATA_DIR, "SILVER")
GOLD_DIR = os.path.join(DATA_DIR, "GOLD")
MODELS_DIR = os.path.join(DATA_DIR, "MODELS")

for p in [BRONZE_DIR, SILVER_DIR, GOLD_DIR, MODELS_DIR]:
    os.makedirs(p, exist_ok=True)

print("DATA_DIR =", DATA_DIR)
print("BRONZE_DIR =", BRONZE_DIR)
print("SILVER_DIR =", SILVER_DIR)
print("GOLD_DIR =", GOLD_DIR)
print("MODELS_DIR =", MODELS_DIR)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 2.2 MB/s eta 0:00:00
Mounted at /content/drive
Spark version: 4.0.2
DATA_DIR = /content/drive/MyDrive/FIFA23_BIGDATA
BRONZE_DIR = /content/drive/MyDrive/FIFA23_BIGDATA/BRONZE
SILVER_DIR = /content/drive/MyDrive/FIFA23_BIGDATA/SILVER
GOLD_DIR = /content/drive/MyDrive/FIFA23_BIGDATA/GOLD
MODELS_DIR = /content/drive/MyDrive/FIFA23_BIGDATA/MODELS


In [2]:
# CELL 2 - KHÓA CỨNG NGUỒN DỮ LIỆU male_players.csv
# [ĐỒNG]
# Mục đích:
# - Không tự dò file nữa
# - Chỉ dùng đúng file male_players.csv làm nguồn chính

SOURCE_PATH = os.path.join(DATA_DIR, "male_players.csv")

if not os.path.exists(SOURCE_PATH):
    raise FileNotFoundError(f"Không tìm thấy file: {SOURCE_PATH}")

print("SOURCE_PATH =", SOURCE_PATH)
print("File size (GB) =", round(os.path.getsize(SOURCE_PATH) / (1024**3), 3))

SOURCE_PATH = /content/drive/MyDrive/FIFA23_BIGDATA/male_players.csv
File size (GB) = 5.25


In [3]:
# CELL 3 - NẠP DỮ LIỆU GỐC VÀO TẦNG BRONZE
# Mục đích:
# - Đưa dữ liệu cầu thủ từ file nguồn vào tầng Bronze
# - Tạo dữ liệu thô để làm đầu vào cho các tầng xử lý sau
#
# Cell này làm được gì:
# - Đọc file male_players.csv bằng Spark
# - Kiểm tra schema dữ liệu
# - Ghi dữ liệu vào Bronze dưới dạng Delta table
#
# Tác dụng trong toàn bài:
# - Đây là bước ingest dữ liệu gốc
# - Là nền tảng để xây dựng kiến trúc Lakehouse cho toàn bộ đồ án

BRONZE_PLAYERS_PATH = os.path.join(BRONZE_DIR, "players")

players_raw_df = (
    spark.read
    .option("header", True)
    .option("multiLine", False)
    .option("escape", '"')
    .option("quote", '"')
    .option("inferSchema", True)
    .csv(SOURCE_PATH)
)

print("Raw row count:", players_raw_df.count())
players_raw_df.printSchema()

(
    players_raw_df.write
    .format("delta")
    .mode("overwrite")
    .save(BRONZE_PLAYERS_PATH)
)

print("Đã ghi Bronze table:", BRONZE_PLAYERS_PATH)

Raw row count: 10003590
root
 |-- player_id: integer (nullable = true)
 |-- player_url: string (nullable = true)
 |-- fifa_version: integer (nullable = true)
 |-- fifa_update: integer (nullable = true)
 |-- fifa_update_date: date (nullable = true)
 |-- short_name: string (nullable = true)
 |-- long_name: string (nullable = true)
 |-- player_positions: string (nullable = true)
 |-- overall: integer (nullable = true)
 |-- potential: integer (nullable = true)
 |-- value_eur: integer (nullable = true)
 |-- wage_eur: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- dob: date (nullable = true)
 |-- height_cm: integer (nullable = true)
 |-- weight_kg: integer (nullable = true)
 |-- league_id: integer (nullable = true)
 |-- league_name: string (nullable = true)
 |-- league_level: integer (nullable = true)
 |-- club_team_id: integer (nullable = true)
 |-- club_name: string (nullable = true)
 |-- club_position: string (nullable = true)
 |-- club_jersey_number: integer (nullable

In [4]:
# CELL 4 - LÀM SẠCH VÀ CHUẨN HÓA DỮ LIỆU CẦU THỦ
# Mục đích:
# - Biến dữ liệu thô thành dữ liệu sạch và nhất quán hơn
# - Chuẩn bị đầu vào cho phân tích và machine learning
#
# Cell này làm được gì:
# - Chọn các cột quan trọng
# - Ép kiểu dữ liệu số
# - Xử lý giá trị thiếu
# - Loại bỏ dữ liệu trùng
# - Tạo thêm một số đặc trưng cơ bản như nhóm vị trí và BMI
#
# Tác dụng trong toàn bài:
# - Đây là bước xây dựng tầng Silver
# - Giúp dữ liệu đủ chất lượng để dùng cho các mô hình và bảng Gold

players_df = spark.read.format("delta").load(BRONZE_PLAYERS_PATH)

candidate_cols = [
    "sofifa_id", "short_name", "long_name", "player_positions",
    "overall", "potential", "age",
    "value_eur", "wage_eur", "height_cm", "weight_kg",
    "club_name", "league_name", "nationality_name", "preferred_foot",
    "pace", "shooting", "passing", "dribbling", "defending", "physic"
]
available_cols = [c for c in candidate_cols if c in players_df.columns]

silver_df = players_df.select(*available_cols)

numeric_cols = [
    "overall", "potential", "age", "value_eur", "wage_eur",
    "height_cm", "weight_kg", "pace", "shooting", "passing",
    "dribbling", "defending", "physic"
]
numeric_cols = [c for c in numeric_cols if c in silver_df.columns]

for c in numeric_cols:
    silver_df = silver_df.withColumn(c, F.col(c).cast("double"))

fill_num = {c: 0.0 for c in numeric_cols}
fill_str = {c: "unknown" for c in silver_df.columns if c not in numeric_cols}
silver_df = silver_df.fillna(fill_num).fillna(fill_str)

if "player_positions" in silver_df.columns:
    silver_df = silver_df.withColumn(
        "position_group",
        F.when(F.lower(F.col("player_positions")).rlike("st|cf|lw|rw|lf|rf"), F.lit("forward"))
         .when(F.lower(F.col("player_positions")).rlike("cam|cm|cdm|lm|rm"), F.lit("midfielder"))
         .when(F.lower(F.col("player_positions")).rlike("cb|lb|rb|lwb|rwb"), F.lit("defender"))
         .when(F.lower(F.col("player_positions")).rlike("gk"), F.lit("goalkeeper"))
         .otherwise(F.lit("other"))
    )
else:
    silver_df = silver_df.withColumn("position_group", F.lit("unknown"))

if set(["height_cm", "weight_kg"]).issubset(set(silver_df.columns)):
    silver_df = silver_df.withColumn(
        "bmi",
        F.when(F.col("height_cm") > 0, F.col("weight_kg") / ((F.col("height_cm") / 100.0) ** 2)).otherwise(F.lit(None))
    )

if set(["potential", "overall"]).issubset(set(silver_df.columns)):
    silver_df = silver_df.withColumn("potential_gap", F.col("potential") - F.col("overall"))

silver_df = silver_df.dropDuplicates()

print("Silver row count:", silver_df.count())
display(silver_df.limit(5).toPandas())

Silver row count: 707595


,short_name,long_name,player_positions,overall,potential,age,value_eur,wage_eur,height_cm,weight_kg,...,preferred_foot,pace,shooting,passing,dribbling,defending,physic,position_group,bmi,potential_gap
0,R. Lewandowski,Robert Lewandowski,ST,91.0,91.0,31.0,111000000.0,240000.0,184.0,80.0,...,Right,78.0,91.0,79.0,85.0,44.0,82.0,forward,23.629490,0.0
1,N. Amrabat,Noureddine Amrabat,"RM, LM",79.0,79.0,33.0,11000000.0,46000.0,180.0,85.0,...,Right,79.0,75.0,77.0,80.0,53.0,79.0,midfielder,26.234568,0.0
2,E. Guichón,Edgar Manuel Guichón,"LB, LM",79.0,79.0,32.0,0.0,0.0,168.0,65.0,...,Left,87.0,58.0,70.0,77.0,74.0,53.0,midfielder,23.030045,0.0
3,Bernard,Bernard Anício Caldeira Duarte,LM,78.0,78.0,27.0,15000000.0,76000.0,164.0,60.0,...,Right,84.0,68.0,73.0,85.0,39.0,43.0,midfielder,22.308150,0.0
4,Roberto Torres,Roberto Torres Morales,"RM, LM, CM",78.0,78.0,31.0,11500000.0,33000.0,178.0,72.0,...,Right,69.0,78.0,79.0,76.0,65.0,72.0,midfielder,22.724403,0.0


In [5]:
# CELL 4B - SILVER/players_silver
# [ĐỒNG]
# Mục đích:
# - Ghi Silver dataframe ra đúng Delta table:
#   SILVER/players_silver
#
# Kết quả mong đợi:
# - SILVER/players_silver/_delta_log
# - SILVER/players_silver/part-*.snappy.parquet

SILVER_PLAYERS_PATH = os.path.join(SILVER_DIR, "players_silver")

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .save(SILVER_PLAYERS_PATH)
)

print("Đã ghi Silver table:", SILVER_PLAYERS_PATH)

Đã ghi Silver table: /content/drive/MyDrive/FIFA23_BIGDATA/SILVER/players_silver


In [6]:
# CELL 5 - TẠO BẢNG THỐNG KÊ TỔNG HỢP CẦU THỦ
# Mục đích:
# - Tạo bảng thống kê mô tả phục vụ phân tích dữ liệu cầu thủ theo nhóm vị trí
#
# Cell này làm được gì:
# - Tính trung bình, min, max của các chỉ số chính theo position_group
# - Tạo bảng tổng hợp dễ đọc và dễ dùng cho báo cáo
#
# Tác dụng trong toàn bài:
# - Cung cấp cái nhìn tổng quan về dữ liệu ở tầng Gold
# - Hỗ trợ giải thích kết quả của các mô hình về sau

GOLD_PLAYER_STATS_PATH = os.path.join(GOLD_DIR, "gold_player_stats")

stats_expr = []
for c in [col for col in ["overall", "potential", "value_eur", "wage_eur", "age"] if col in silver_df.columns]:
    stats_expr.extend([
        F.avg(c).alias(f"avg_{c}"),
        F.min(c).alias(f"min_{c}"),
        F.max(c).alias(f"max_{c}")
    ])

gold_player_stats_df = (
    silver_df.groupBy("position_group")
    .agg(*stats_expr)
)

(
    gold_player_stats_df.write
    .format("delta")
    .mode("overwrite")
    .save(GOLD_PLAYER_STATS_PATH)
)

print("Đã ghi:", GOLD_PLAYER_STATS_PATH)
display(gold_player_stats_df.toPandas().head())

Đã ghi: /content/drive/MyDrive/FIFA23_BIGDATA/GOLD/gold_player_stats


,position_group,avg_overall,min_overall,max_overall,avg_potential,min_potential,max_potential,avg_value_eur,min_value_eur,max_value_eur,avg_wage_eur,min_wage_eur,max_wage_eur,avg_age,min_age,max_age
0,goalkeeper,64.016231,40.0,92.0,69.464484,40.0,94.0,1.532689e+06,0.0,120000000.0,7476.236859,0.0,300000.0,25.440403,16.0,55.0
1,forward,66.418655,40.0,94.0,71.674691,48.0,98.0,2.903853e+06,0.0,194000000.0,12640.405211,0.0,575000.0,24.426048,15.0,54.0
2,defender,65.752688,40.0,91.0,70.743558,46.0,93.0,1.956504e+06,0.0,114000000.0,10342.064785,0.0,375000.0,24.965343,15.0,43.0
3,midfielder,65.829789,35.0,91.0,71.275408,40.0,94.0,2.327506e+06,0.0,129000000.0,10739.794193,0.0,425000.0,24.388903,15.0,43.0


In [7]:
# CELL 6 - CHUẨN BỊ SAMPLE PANDAS CHO ML
# [BẠC]
# Mục đích:
# - Lấy sample vừa phải từ Silver để train model local trong notebook
# - Tránh nổ RAM

MAX_ROWS = 50000

silver_sample_pdf = silver_df.limit(MAX_ROWS).toPandas().copy()
print("silver_sample_pdf shape =", silver_sample_pdf.shape)
display(silver_sample_pdf.head())

silver_sample_pdf shape = (50000, 23)


,short_name,long_name,player_positions,overall,potential,age,value_eur,wage_eur,height_cm,weight_kg,...,preferred_foot,pace,shooting,passing,dribbling,defending,physic,position_group,bmi,potential_gap
0,R. Lewandowski,Robert Lewandowski,ST,91.0,91.0,31.0,111000000.0,240000.0,184.0,80.0,...,Right,78.0,91.0,79.0,85.0,44.0,82.0,forward,23.629490,0.0
1,N. Amrabat,Noureddine Amrabat,"RM, LM",79.0,79.0,33.0,11000000.0,46000.0,180.0,85.0,...,Right,79.0,75.0,77.0,80.0,53.0,79.0,midfielder,26.234568,0.0
2,E. Guichón,Edgar Manuel Guichón,"LB, LM",79.0,79.0,32.0,0.0,0.0,168.0,65.0,...,Left,87.0,58.0,70.0,77.0,74.0,53.0,midfielder,23.030045,0.0
3,Bernard,Bernard Anício Caldeira Duarte,LM,78.0,78.0,27.0,15000000.0,76000.0,164.0,60.0,...,Right,84.0,68.0,73.0,85.0,39.0,43.0,midfielder,22.308150,0.0
4,Roberto Torres,Roberto Torres Morales,"RM, LM, CM",78.0,78.0,31.0,11500000.0,33000.0,178.0,72.0,...,Right,69.0,78.0,79.0,76.0,65.0,72.0,midfielder,22.724403,0.0


In [8]:
# CELL 7 - MODEL PREP CHO SUPERVISED ML
# [ĐỒNG]
# Mục đích:
# - Chuẩn bị feature cho:
#   + value prediction
#   + overall prediction
#   + clustering
#
# Lưu ý:
# - Dùng sample để train model
# - Sau đó ghi prediction/result ra GOLD

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD

df_model = silver_sample_pdf.copy()

id_candidates = ["short_name", "long_name", "sofifa_id"]
id_col = next((c for c in id_candidates if c in df_model.columns), None)

feature_drop = [c for c in [id_col, "short_name", "long_name", "sofifa_id"] if c in df_model.columns]
base_features_df = df_model.drop(columns=feature_drop, errors="ignore")

numeric_cols = base_features_df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = base_features_df.select_dtypes(include=["object"]).columns.tolist()

prep = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), numeric_cols),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), categorical_cols)
    ]
)

print("numeric_cols:", numeric_cols)
print("categorical_cols:", categorical_cols)

numeric_cols: ['overall', 'potential', 'age', 'value_eur', 'wage_eur', 'height_cm', 'weight_kg', 'pace', 'shooting', 'passing', 'dribbling', 'defending', 'physic', 'bmi', 'potential_gap']
categorical_cols: ['player_positions', 'club_name', 'league_name', 'nationality_name', 'preferred_foot', 'position_group']


In [11]:
# CELL 8 - DỰ ĐOÁN GIÁ TRỊ CẦU THỦ BẰNG RANDOM FOREST
# Mục đích:
# - Xây dựng mô hình dự đoán giá trị cầu thủ từ các đặc trưng chuyên môn và thể chất
#
# Cell này làm được gì:
# - Chuẩn bị dữ liệu huấn luyện cho bài toán hồi quy
# - Huấn luyện mô hình Random Forest Regressor
# - Tính các chỉ số đánh giá như MAE, RMSE và R²
# - Lưu mô hình đã huấn luyện và lưu bảng dự đoán
#
# Tác dụng trong toàn bài:
# - Đây là mô hình supervised chính của đồ án
# - Giúp kiểm tra khả năng học mối quan hệ giữa hồ sơ cầu thủ và giá trị kinh tế

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor as SparkRFRegressor
from pyspark.ml import Pipeline as SparkPipeline

if "value_eur" not in df_model.columns:
    raise ValueError("Thiếu cột value_eur để train value model.")

# =========================
# 1. TẠO DATA CHO SKLEARN
# =========================
target_value = "value_eur"

# Drop target khỏi feature
X_value = df_model.drop(columns=[target_value], errors="ignore").copy()
y_value = df_model[target_value].astype(float)

# Nếu có cột id thì bỏ ra khỏi feature
drop_id_cols = [c for c in [id_col, "short_name", "long_name", "sofifa_id"] if c and c in X_value.columns]
X_value = X_value.drop(columns=drop_id_cols, errors="ignore")

# Xác định lại cột numeric/categorical RIÊNG cho bài toán value
num_cols_value = X_value.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_value = X_value.select_dtypes(include=["object"]).columns.tolist()

X_train_v, X_test_v, y_train_v, y_test_v = train_test_split(
    X_value, y_value, test_size=0.2, random_state=42
)

# Preprocessor RIÊNG cho value model
prep_value = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols_value),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols_value)
    ]
)

value_pipeline = Pipeline(steps=[
    ("prep", prep_value),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

value_pipeline.fit(X_train_v, y_train_v)
pred_v = value_pipeline.predict(X_test_v)

value_mae = mean_absolute_error(y_test_v, pred_v)
value_rmse = np.sqrt(mean_squared_error(y_test_v, pred_v))
value_r2 = r2_score(y_test_v, pred_v)

print("Value RF MAE =", round(value_mae, 4))
print("Value RF RMSE =", round(value_rmse, 4))
print("Value RF R2 =", round(value_r2, 4))

# =========================
# 2. TRAIN SPARK MODEL ĐỂ SAVE ĐÚNG STRUCTURE
# =========================
spark_value_cols = [
    c for c in [
        "overall", "potential", "age", "wage_eur", "height_cm", "weight_kg",
        "pace", "shooting", "passing", "dribbling", "defending", "physic"
    ] if c in silver_df.columns
]

if "value_eur" not in silver_df.columns:
    raise ValueError("Thiếu cột value_eur trong silver_df.")

spark_value_df = silver_df.select(*spark_value_cols, "value_eur").fillna(0.0)

assembler_value = VectorAssembler(inputCols=spark_value_cols, outputCol="features")
rf_value = SparkRFRegressor(featuresCol="features", labelCol="value_eur", numTrees=100, seed=42)
spark_value_pipeline = SparkPipeline(stages=[assembler_value, rf_value])

spark_value_model = spark_value_pipeline.fit(spark_value_df)

VALUE_MODEL_PATH = os.path.join(MODELS_DIR, "value_predict_rf")
spark_value_model.write().overwrite().save(VALUE_MODEL_PATH)

# =========================
# 3. GHI GOLD PREDICTIONS
# =========================
gold_value_pdf = X_test_v.copy()
gold_value_pdf["actual_value_eur"] = y_test_v.values
gold_value_pdf["predicted_value_eur"] = pred_v

GOLD_VALUE_ML_PATH = os.path.join(GOLD_DIR, "gold_value_ml")
spark.createDataFrame(gold_value_pdf).write.format("delta").mode("overwrite").save(GOLD_VALUE_ML_PATH)

print("Đã ghi model:", VALUE_MODEL_PATH)
print("Đã ghi gold predictions:", GOLD_VALUE_ML_PATH)

Value RF MAE = 362639.978
Value RF RMSE = 1412237.9522
Value RF R2 = 0.9513
Đã ghi model: /content/drive/MyDrive/FIFA23_BIGDATA/MODELS/value_predict_rf
Đã ghi gold predictions: /content/drive/MyDrive/FIFA23_BIGDATA/GOLD/gold_value_ml


In [14]:
# CELL 9 - DỰ ĐOÁN CHỈ SỐ OVERALL CỦA CẦU THỦ
# Mục đích:
# - Xây dựng mô hình dự đoán chỉ số tổng quát của cầu thủ
#
# Cell này làm được gì:
# - Chuẩn bị dữ liệu cho bài toán hồi quy overall
# - Huấn luyện Random Forest Regressor
# - Tính các chỉ số đánh giá hiệu năng mô hình
# - Tạo bảng dự đoán actual và predicted overall
#
# Tác dụng trong toàn bài:
# - Giúp kiểm tra mức độ mô hình có thể học được chất lượng tổng quát của cầu thủ
# - Bổ sung thêm một bài toán machine learning ngoài value prediction

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor

if "overall" not in df_model.columns:
    raise ValueError("Thiếu cột overall để train overall model.")

target_overall = "overall"

X_overall = df_model.drop(columns=[target_overall], errors="ignore").copy()
y_overall = df_model[target_overall].astype(float)

drop_id_cols = [c for c in [id_col, "short_name", "long_name", "sofifa_id"] if c and c in X_overall.columns]
X_overall = X_overall.drop(columns=drop_id_cols, errors="ignore")

num_cols_overall = X_overall.select_dtypes(include=[np.number]).columns.tolist()
cat_cols_overall = X_overall.select_dtypes(include=["object"]).columns.tolist()

X_train_o, X_test_o, y_train_o, y_test_o = train_test_split(
    X_overall, y_overall, test_size=0.2, random_state=42
)

prep_overall = ColumnTransformer(
    transformers=[
        ("num", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), num_cols_overall),
        ("cat", Pipeline(steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols_overall)
    ]
)

overall_pipeline = Pipeline(steps=[
    ("prep", prep_overall),
    ("model", RandomForestRegressor(
        n_estimators=200,
        random_state=42,
        n_jobs=-1
    ))
])

overall_pipeline.fit(X_train_o, y_train_o)
pred_o = overall_pipeline.predict(X_test_o)

overall_mae = mean_absolute_error(y_test_o, pred_o)
overall_rmse = np.sqrt(mean_squared_error(y_test_o, pred_o))
overall_r2 = r2_score(y_test_o, pred_o)

print("Overall RF MAE =", round(overall_mae, 4))
print("Overall RF RMSE =", round(overall_rmse, 4))
print("Overall RF R2 =", round(overall_r2, 4))

gold_overall_pdf = X_test_o.copy()
gold_overall_pdf["actual_overall"] = y_test_o.values
gold_overall_pdf["predicted_overall"] = pred_o

GOLD_OVERALL_ML_PATH = os.path.join(GOLD_DIR, "gold_overall_ml")
spark.createDataFrame(gold_overall_pdf).write.format("delta").mode("overwrite").save(GOLD_OVERALL_ML_PATH)

print("Đã ghi:", GOLD_OVERALL_ML_PATH)

Overall RF MAE = 0.0807
Overall RF RMSE = 0.2089
Overall RF R2 = 0.9991
Đã ghi: /content/drive/MyDrive/FIFA23_BIGDATA/GOLD/gold_overall_ml


In [15]:
# CELL 10 - PHÂN CỤM CẦU THỦ THEO HỒ SƠ ĐẶC TRƯNG
# Mục đích:
# - Chia cầu thủ thành các nhóm có đặc điểm tương đồng
#
# Cell này làm được gì:
# - Chọn các biến quan trọng cho bài toán phân cụm
# - Huấn luyện mô hình KMeans
# - Gán nhãn cụm cho từng cầu thủ
# - Tạo bảng mô tả trung bình của từng cụm
#
# Tác dụng trong toàn bài:
# - Giúp khám phá cấu trúc ẩn của dữ liệu cầu thủ
# - Hỗ trợ phân tích nhóm cầu thủ và nhận diện các kiểu cầu thủ khác nhau

cluster_features = [c for c in ["overall", "potential", "age", "value_eur", "wage_eur", "pace", "shooting", "passing", "dribbling", "defending", "physic"] if c in df_model.columns]
cluster_pdf = df_model[[c for c in cluster_features if c in df_model.columns] + [c for c in [id_col, "position_group"] if c in df_model.columns]].copy()

cluster_X = cluster_pdf[cluster_features].fillna(0.0)

kmeans = KMeans(n_clusters=5, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(cluster_X)

cluster_result_pdf = cluster_pdf.copy()
cluster_result_pdf["cluster_id"] = cluster_labels

cluster_summary_pdf = (
    cluster_result_pdf.groupby("cluster_id")[cluster_features]
    .mean()
    .reset_index()
)

GOLD_CLUSTER_RESULT_PATH = os.path.join(GOLD_DIR, "gold_cluster_result")
GOLD_CLUSTER_ML_PATH = os.path.join(GOLD_DIR, "gold_cluster_ml")

spark.createDataFrame(cluster_result_pdf).write.format("delta").mode("overwrite").save(GOLD_CLUSTER_RESULT_PATH)
spark.createDataFrame(cluster_summary_pdf).write.format("delta").mode("overwrite").save(GOLD_CLUSTER_ML_PATH)

print("Đã ghi:", GOLD_CLUSTER_RESULT_PATH)
print("Đã ghi:", GOLD_CLUSTER_ML_PATH)
display(cluster_summary_pdf.head())

Đã ghi: /content/drive/MyDrive/FIFA23_BIGDATA/GOLD/gold_cluster_result
Đã ghi: /content/drive/MyDrive/FIFA23_BIGDATA/GOLD/gold_cluster_ml


,cluster_id,overall,potential,age,value_eur,wage_eur,pace,shooting,passing,dribbling,defending,physic
0,0,64.281615,70.188934,24.689317,1.017775e+06,4685.673356,60.825890,45.785918,50.063589,55.020786,45.043986,57.309662
1,1,79.766923,82.866154,25.756154,2.024577e+07,52794.192308,69.217692,62.483846,67.023077,71.187692,56.907692,67.215385
2,2,83.639903,86.625304,25.632603,4.360462e+07,112690.997567,73.245742,67.975669,72.377129,76.523114,58.508516,68.841849
3,3,75.241400,78.648749,25.929437,7.520504e+06,25507.398358,67.774238,58.817240,63.245895,67.777365,54.442338,65.806489
4,4,88.000000,89.602941,26.779412,9.044853e+07,211808.823529,75.558824,73.941176,73.676471,78.691176,48.441176,69.132353


In [16]:
# CELL 11 - FINAL CHECK
# [ĐỒNG]
# Mục đích:
# - Kiểm tra đúng structure cuối cùng
# - Liệt kê folder trong BRONZE / GOLD / MODELS

final_checks = {
    "source_path": SOURCE_PATH,
    "bronze_players_path": BRONZE_PLAYERS_PATH,
    "silver_players_path": SILVER_PLAYERS_PATH,
    "gold_player_stats_path": GOLD_PLAYER_STATS_PATH,
    "gold_value_ml_path": GOLD_VALUE_ML_PATH,
    "gold_overall_ml_path": GOLD_OVERALL_ML_PATH,
    "gold_cluster_result_path": GOLD_CLUSTER_RESULT_PATH,
    "gold_cluster_ml_path": GOLD_CLUSTER_ML_PATH,
    "value_model_path": VALUE_MODEL_PATH,
    "value_rf_mae": float(value_mae),
    "value_rf_rmse": float(value_rmse),
    "value_rf_r2": float(value_r2),
    "overall_rf_mae": float(overall_mae),
    "overall_rf_rmse": float(overall_rmse),
    "overall_rf_r2": float(overall_r2),
}

final_json = os.path.join(DATA_DIR, "final_checks_exact_structure.json")
with open(final_json, "w", encoding="utf-8") as f:
    json.dump(final_checks, f, ensure_ascii=False, indent=2)

print(json.dumps(final_checks, ensure_ascii=False, indent=2))

print("\nBRONZE:")
print(os.listdir(BRONZE_DIR))

print("\nSILVER:")
print(os.listdir(SILVER_DIR))

print("\nGOLD:")
print(os.listdir(GOLD_DIR))

print("\nMODELS:")
print(os.listdir(MODELS_DIR))

{
  "source_path": "/content/drive/MyDrive/FIFA23_BIGDATA/male_players.csv",
  "bronze_players_path": "/content/drive/MyDrive/FIFA23_BIGDATA/BRONZE/players",
  "silver_players_path": "/content/drive/MyDrive/FIFA23_BIGDATA/SILVER/players_silver",
  "gold_player_stats_path": "/content/drive/MyDrive/FIFA23_BIGDATA/GOLD/gold_player_stats",
  "gold_value_ml_path": "/content/drive/MyDrive/FIFA23_BIGDATA/GOLD/gold_value_ml",
  "gold_overall_ml_path": "/content/drive/MyDrive/FIFA23_BIGDATA/GOLD/gold_overall_ml",
  "gold_cluster_result_path": "/content/drive/MyDrive/FIFA23_BIGDATA/GOLD/gold_cluster_result",
  "gold_cluster_ml_path": "/content/drive/MyDrive/FIFA23_BIGDATA/GOLD/gold_cluster_ml",
  "value_model_path": "/content/drive/MyDrive/FIFA23_BIGDATA/MODELS/value_predict_rf",
  "value_rf_mae": 362639.978,
  "value_rf_rmse": 1412237.952185571,
  "value_rf_r2": 0.9513180070044756,
  "overall_rf_mae": 0.08069550000000002,
  "overall_rf_rmse": 0.20889351952609728,
  "overall_rf_r2": 0.9990751897